In [1]:
import os
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import torch.nn.functional as F
from torchvision.utils import save_image

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

Model Architecture is :

784 inputs ----> 400 Hidden layer ----> 20,20 mean,variance ----> add them ----> 400 decode hidden layer
----> 784 outputs

In [4]:
#defining hyperparameters

In [5]:
image_size = 784 # 28*28 for MNIST
hidden_dim = 400
latent_dim = 20
batch_size = 128
epochs = 10

train_dataset = torchvision.datasets.MNIST(root='mnist_for_vae',
                                          train=True,
                                          transform=transforms.ToTensor(),
                                          download=True)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)
test_dataset = torchvision.datasets.MNIST(root='mnist_for_vae',
                                          train=False,
                                          transform=transforms.ToTensor())
test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                          batch_size=batch_size,
                                          shuffle=True)

# making directory for saving output images
directory = "VAE_results"
if not os.path.exists(directory) : 
    os.makedirs(directory)

In [9]:
# VAE Model
class VAE(nn.Module) : 
    def __init__(self) : 
        super(VAE,self).__init__()
        self.fc1 = nn.Linear(image_size,hidden_dim)
        self.fc2_mean = nn.Linear(hidden_dim,latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim,latent_dim)
        self.fc3 = nn.Linear(latent_dim,hidden_dim)
        self.fc4 = nn.Linear(hidden_dim,image_size)
    def encode(self,x) : 
        h = F.relu(self.fc1(x))
        mu = self.fc2_mean(h)
        log_var = self.fc2_logvar(h)
        return mu, log_var
    def reparameterize(self, mu, logvar) : 
        std = torch.exp(logvar/2)
        eps = torch.randn_like(std)
        return mu + eps*std
    def decode(self,z) : 
        h = F.relu(self.fc3(z))
        out = torch.sigmoid(self.fc4(h))
        return out
    def forward(self,x) : 
        mu, logvar = self.encode(x.view(-1,image_size))
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        return reconstructed, mu, logvar

# defining model and optimizer
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

In [12]:
# defining loss function, train and test algo
def loss_fn(reconstructed_image, original_image, mu, logvar) : 
    bce = F.binary_cross_entropy(reconstructed_image,original_image.view(-1,784), reduction = 'sum')
    kld = 0.5*torch.sum(logvar.exp() + mu.pow(2) - 1 - logvar)
    return bce+kld

def train(epoch) : 
    model.train()
    train_loss = 0
    for i, (images,_) in enumerate(train_loader) : 
        images = images.to(device)
        reconstructed, mu, logvar = model(images)
        loss = loss_fn(reconstructed, images, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if i%100 == 0 : 
            print(f"epoch {epoch} [Batch {i}/{len(train_loader)}], loss = {loss.item()/len(images) :.3f}")
    print(f"Epoch {epoch}, Avg. Loss = {train_loss/len(train_loader.dataset):.3f}")

def test(epoch) : 
    model.eval()
    test_loss = 0
    with torch.no_grad() : 
        for batch_index, (images,_) in enumerate(test_loader) : 
            images = images.to(device)
            reconstructed, mu, logvar = model(images)
            loss = loss_fn(reconstructed, images, mu, logvar)
            test_loss += loss.item()
            if batch_index == 0 : 
                comparison = torch.cat([images[:5], reconstructed.view(batch_size,1,28,28)[:5]])
                save_image(comparison.cpu(),"VAE_results/recontruction_"+str(epoch)+".png",nrow = 5)

In [14]:
# defining main function
for epoch in range(epochs) : 
    train(epoch)
    test(epoch)
    with torch.no_grad() : 
        # we want to get rid of the encoder and sample some values from the gaussian distribution and feed
        # them to the decoder to generate new images of same style but not in the dataset !!!
        sample = torch.randn(64,20).to(device)
        generated = model.decode(sample).cpu()
        save_image(generated.view(64,1,28,28), 'VAE_results/sample_'+str(epoch)+'.png')

epoch 0 [Batch 0/469], loss = 114.771
epoch 0 [Batch 100/469], loss = 114.767
epoch 0 [Batch 200/469], loss = 117.315
epoch 0 [Batch 300/469], loss = 113.465
epoch 0 [Batch 400/469], loss = 111.294
Epoch 0, Avg. Loss = 114.418
epoch 1 [Batch 0/469], loss = 108.610
epoch 1 [Batch 100/469], loss = 111.542
epoch 1 [Batch 200/469], loss = 113.247
epoch 1 [Batch 300/469], loss = 107.043
epoch 1 [Batch 400/469], loss = 111.978
Epoch 1, Avg. Loss = 111.509
epoch 2 [Batch 0/469], loss = 109.422
epoch 2 [Batch 100/469], loss = 109.066
epoch 2 [Batch 200/469], loss = 107.804
epoch 2 [Batch 300/469], loss = 105.451
epoch 2 [Batch 400/469], loss = 109.641
Epoch 2, Avg. Loss = 109.803
epoch 3 [Batch 0/469], loss = 111.812
epoch 3 [Batch 100/469], loss = 109.239
epoch 3 [Batch 200/469], loss = 111.746
epoch 3 [Batch 300/469], loss = 108.904
epoch 3 [Batch 400/469], loss = 103.369
Epoch 3, Avg. Loss = 108.641
epoch 4 [Batch 0/469], loss = 113.390
epoch 4 [Batch 100/469], loss = 109.872
epoch 4 [Batch